
# Adult Income — Simple, Commented ML Lifecycle

This notebook is intentionally **simple** and **well-explained**. It follows the required steps:

1. Data loading and quick EDA  
2. Clean preprocessing (no leakage)  
3. Two baseline models (Logistic Regression, Random Forest)  
4. Evaluation with key metrics & plots  
5. Interpretability (Permutation Importance)  
6. Save final model  


## 0) Setup & Imports

In [1]:

import os, pathlib, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
import joblib

BASE = pathlib.Path('').resolve()
DATA = BASE / 'data'
REPORTS = BASE / 'reports' / 'figures'
MODELS = BASE / 'models'
REPORTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

print('Data folder:', DATA)


Data folder: /home/AnirSaddik/Desktop/Year3/Semester1/Data5/Term1/Project/to-check-poject/ML_Lifecycle_Project_AdultIncome/notebooks/data


## 1) Load the Adult dataset

In [7]:
# === SIMPLE ADULT DATA LOADER (choose ONE file path) ===
import pandas as pd
from pathlib import Path

# >>> EDIT THIS to match your file <<<
# Examples:
# DATA_PATH = "data/adult.csv"
# DATA_PATH = "data/adult.data"
# DATA_PATH = "data/adult_test.csv"
DATA_PATH = "data/adult.csv"   # <-- change if your filename is different

# UCI adult column order (used when file has no header, e.g. adult.data)
COLS = [
    "age","workclass","fnlwgt","education","education-num",
    "marital-status","occupation","relationship","race","sex",
    "capital-gain","capital-loss","hours-per-week","native-country","income"
]

def load_adult(path: str) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found at: {path.resolve()}")

    # Case 1: UCI .data without header
    if path.suffix.lower() == ".data":
        df = pd.read_csv(path, header=None, names=COLS, na_values="?", skipinitialspace=True)
    else:
        # Case 2: CSV with headers (Kaggle/other variants)
        df = pd.read_csv(path, na_values="?")
        # If the target isn't named 'income', rename common alternatives
        if "income" not in df.columns:
            for c in df.columns:
                if str(c).lower() in {"class", "target", "salary"}:
                    df = df.rename(columns={c: "income"})
                    break

    # Standardize common column name variations
    df = df.rename(columns={
        "capital_gain": "capital-gain",
        "capital_loss": "capital-loss",
        "hours_per_week": "hours-per-week",
        "education_num": "education-num",
        "marital_status": "marital-status",
        "native_country": "native-country",
    })

    # Light cleanup: trim strings
    for c in df.select_dtypes(include="object").columns:
        df[c] = df[c].astype(str).str.strip()

    # Make sure the target exists
    if "income" not in df.columns:
        raise ValueError("Could not find the target column 'income'. Please check your file or rename it.")

    return df

df = load_adult(DATA_PATH)
print(f"Loaded {len(df)} rows, {df.shape[1]} columns from {DATA_PATH}")
df.head()


FileNotFoundError: File not found at: /home/AnirSaddik/Desktop/Year3/Semester1/Data5/Term1/Project/to-check-poject/ML_Lifecycle_Project_AdultIncome/notebooks/data/adult.csv

## 2) Quick EDA & Basic Cleaning

In [ ]:

# Strip whitespace from strings; replace '?' with NaN (common placeholder)
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.strip()
df = df.replace('?', np.nan)

# Target normalization: exact two classes
y = df['income'].astype(str).str.strip().apply(lambda v: '>50K' if v in {'>50K','>50K.'} else '<=50K')
X = df.drop(columns=['income'])

# Identify numeric vs categorical
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print('Numeric:', numeric_cols)
print('Categorical (first 10):', categorical_cols[:10])

# Class balance
print("\nClass balance:")
print(y.value_counts(normalize=True))

# Visualize class distribution
y.value_counts(normalize=True).plot(kind='bar', title='Target distribution')
plt.ylabel('proportion'); plt.tight_layout(); plt.savefig(REPORTS/'target_distribution.png'); plt.show()



**Why these steps?**  
- Cleaning ensures consistent categories and proper missing values.  
- Splitting numerics/categoricals lets us apply the right transforms.  
- The class balance tells us accuracy alone can mislead; we'll track **precision/recall/F1/ROC-AUC**.


## 3) Leakage-Safe Preprocessing

In [ ]:

# Train/test split (stratified to preserve class ratio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Pipelines for numeric and categorical features
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine with ColumnTransformer
# NOTE: sparse_threshold=0.0 makes output dense; avoids sparse/dense surprises.
preprocess = ColumnTransformer(
    [
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols)
    ],
    sparse_threshold=0.0
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## 4) Two Simple Baseline Models

In [ ]:

logreg = Pipeline([("preprocess", preprocess),
                   ("clf", LogisticRegression(max_iter=500, class_weight="balanced"))])

rf = Pipeline([("preprocess", preprocess),
               ("clf", RandomForestClassifier(n_estimators=300, random_state=42,
                                             n_jobs=-1, class_weight="balanced"))])

logreg.fit(X_train, y_train)
rf.fit(X_train, y_train)

best_models = {"logreg": logreg, "rf": rf}


## 5) Evaluate & Plot

In [ ]:

def evaluate_model(model, X_te, y_te, label):
    y_pred = model.predict(X_te)
    # Score for ROC (probabilities preferred; fallback to decision_function)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_te)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_te)
    else:
        y_score = None

    acc = accuracy_score(y_te, y_pred)
    pre = precision_score(y_te, y_pred, pos_label=">50K")
    rec = recall_score(y_te, y_pred, pos_label=">50K")
    f1  = f1_score(y_te, y_pred, pos_label=">50K")
    roc = roc_auc_score(y_te, y_score, pos_label=">50K") if y_score is not None else np.nan

    print(f"{label}: ACC={acc:.3f} PRE={pre:.3f} REC={rec:.3f} F1={f1:.3f} ROC-AUC={roc:.3f}")
    print(classification_report(y_te, y_pred))

    cm = confusion_matrix(y_te, y_pred, labels=["<=50K", ">50K"])
    ConfusionMatrixDisplay(cm, display_labels=["<=50K", ">50K"]).plot(values_format='d')
    plt.title(f"Confusion Matrix — {label}"); plt.tight_layout()
    plt.savefig(REPORTS / f"cm_{label}.png"); plt.show()

    if y_score is not None:
        RocCurveDisplay.from_predictions(y_te, y_score, pos_label=">50K")
        plt.title(f"ROC Curve — {label}"); plt.tight_layout()
        plt.savefig(REPORTS / f"roc_{label}.png"); plt.show()

    return {"acc":acc, "pre":pre, "rec":rec, "f1":f1, "roc_auc":roc}

scores = {}
for name, model in best_models.items():
    scores[name] = evaluate_model(model, X_test, y_test, name)

pd.DataFrame(scores).T.sort_values('roc_auc', ascending=False)


## 6) Interpretability: Permutation Importance

In [ ]:

# Pick best by ROC-AUC (fallback to F1 if NaN)
def rank_key(k):
    s = scores[k]
    roc = s['roc_auc'] if not np.isnan(s['roc_auc']) else -1.0
    return (roc, s['f1'])

best_name = max(scores, key=rank_key)
best_final = best_models[best_name]
print('Best model:', best_name)

perm = permutation_importance(best_final, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
feat_names = best_final.named_steps['preprocess'].get_feature_names_out()
imp = pd.Series(perm.importances_mean, index=feat_names).sort_values(ascending=False).head(20)

ax = imp.plot(kind='barh'); ax.invert_yaxis()
plt.title('Top Permutation Importances'); plt.tight_layout()
plt.savefig(REPORTS / 'permutation_importances.png'); plt.show()

imp.to_frame('importance').head(20)


## 7) Save Final Model

In [ ]:

joblib.dump(best_final, MODELS / 'final_model.joblib')
MODELS / 'final_model.joblib'



## 8) What else to try (options, kept simple)
- Add **Gradient Boosting** (e.g., `XGBoost/LightGBM/CatBoost`) for a potential boost.
- Try a small **RandomizedSearchCV** on a few key hyperparameters.
- Consider **calibration** if you care about calibrated probabilities.
- Examine **fairness** across groups (e.g., by `sex` / `race`) by comparing metrics.
- Deploy with a tiny **FastAPI** endpoint that loads `models/final_model.joblib`.
